# Corrective RAG (CRAG) with LangGraph

Plain RAG trusts whatever the retriever returns. CRAG adds a quality check after retrieval and changes what it does depending on the result:

**retrieve → evaluate → (refine | rewrite query → web search → refine) → generate**

Based on *Corrective Retrieval Augmented Generation* (Yan et al., 2024, arXiv:2401.15884). This is my own implementation and it differs from the paper in a few ways (listed in the README). The baseline is in `01_basic_rag_baseline.ipynb`.

## 1. Imports
The same libraries as the baseline, plus Tavily for web search and Pydantic for structured LLM output.

In [1]:
import re
from pathlib import Path
from typing import List, TypedDict

from dotenv import load_dotenv
from pydantic import BaseModel

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import END, START, StateGraph

C:\Users\MAHEK\AppData\Local\Temp\ipykernel_2632\2485810828.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## 2. Configuration
API keys are loaded from `.env`. Two thresholds decide the evaluator's verdict: if any chunk scores above `UPPER_TH` (0.7) the retrieval is **Correct**; if every chunk scores at or below `LOWER_TH` (0.3) it is **Incorrect**; anything in between is **Ambiguous**. I chose these values myself and did not tune them. I use two models: `gpt-oss-120b` for the final answer and the smaller `gpt-oss-20b` for the many cheap steps (scoring, filtering, query rewriting). `MAX_CONC = 1` runs batch calls one at a time to stay within Groq's free-tier rate limits.

In [2]:
load_dotenv()  # reads GOOGLE_API_KEY, GROQ_API_KEY, TAVILY_API_KEY from .env

embedding_model = "gemini-embedding-001"
chat_model = "openai/gpt-oss-120b"
fast_model = "openai/gpt-oss-20b"
MAX_CONC = 1    
docs_dir = next((p for p in [Path("documents"), Path("../documents")] if p.exists()), Path("documents"))
TOP_K = 4
UPPER_TH = 0.7   # any chunk above this  -> CORRECT
LOWER_TH = 0.3   # all chunks at/below   -> INCORRECT, otherwise AMBIGUOUS

## 3. Load PDFs and split into chunks
Every PDF in `documents/` is loaded. Chunk overlap keeps sentences from being cut off at chunk boundaries.

In [3]:
pdf_paths = sorted(docs_dir.glob("*.pdf"))
if not pdf_paths:
    raise FileNotFoundError(f"Put at least one PDF in {docs_dir.resolve()}")

docs = [page for p in pdf_paths for page in PyPDFLoader(str(p)).load()]
print(f"Loaded {len(docs)} pages from {len(pdf_paths)} PDF(s)")

chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)
for c in chunks:  # drop characters PDFs sometimes produce that crash the embedding API
    c.page_content = c.page_content.encode("utf-8", "ignore").decode("utf-8", "ignore")

Loaded 16 pages from 1 PDF(s)


## 4. Embeddings, vector store, retriever, models
The retrieval setup is the same as in the baseline, so the comparison is fair. Two Groq models are created: `llm` for the final answer and `fast_llm` for scoring, filtering and rewriting. Both have retries enabled and low reasoning effort because the free tier limits tokens per minute. Tavily is the web search tool.

In [4]:
embeddings = GoogleGenerativeAIEmbeddings(model=embedding_model)
vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": TOP_K})

llm = ChatGroq(model=chat_model, temperature=0, max_retries=8, reasoning_effort="low")
fast_llm = ChatGroq(model=fast_model, temperature=0, max_retries=8, reasoning_effort="low")
tavily = TavilySearch(max_results=5)

## 5. Graph state
LangGraph nodes share information through this state. Compared with the baseline, it also holds the evaluator's verdict, the web results, the rewritten query and the refined context.

In [5]:
class State(TypedDict, total=False):
    question: str
    docs: List[Document]
    good_docs: List[Document]
    web_docs: List[Document]
    verdict: str
    reason: str
    web_query: str
    strips: List[str]
    kept_strips: List[str]
    refined_context: str
    answer: str

## 6. Retrieve node

In [6]:
def retrieve(state: State) -> dict:
    return {"docs": retriever.invoke(state["question"])}

## 7. Retrieval evaluator (LLM judge)
For each retrieved chunk, an LLM returns a relevance score between 0 and 1 and a short reason. I use structured output (a Pydantic model) so the score is always a number. The scores decide the verdict: **CORRECT** (some chunk is clearly relevant), **INCORRECT** (nothing is relevant) or **AMBIGUOUS** (somewhere in between). The paper trains a small T5 model for this step. I used an LLM judge instead because I had no labelled data to train on. The downside is that an LLM judge can be inconsistent, and I have not validated it.

In [7]:
class DocEvalScore(BaseModel):
    score: float
    reason: str

doc_eval_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a strict retrieval evaluator for RAG.\n"
     "You will be given ONE retrieved chunk and a question.\n"
     "Return a relevance score in [0.0, 1.0].\n"
     "- 1.0: chunk alone is sufficient to answer fully/mostly\n"
     "- 0.0: chunk is irrelevant\n"
     "Be conservative with high scores.\n"
     "Also return a short reason.\n"
     "Output JSON only."),
    ("human", "Question: {question}\n\nChunk:\n{chunk}"),
])
doc_eval_chain = doc_eval_prompt | fast_llm.with_structured_output(DocEvalScore)

def eval_each_doc_node(state: State) -> dict:
    results = doc_eval_chain.batch(
        [{"question": state["question"], "chunk": d.page_content} for d in state["docs"]],
        config={"max_concurrency": MAX_CONC},
    )
    scores = [min(max(r.score, 0.0), 1.0) for r in results]          # clamp to [0, 1]
    good_docs = [d for d, s in zip(state["docs"], scores) if s > LOWER_TH]
    shown = [round(s, 2) for s in scores]

    if any(s > UPPER_TH for s in scores):
        return {"good_docs": good_docs, "verdict": "CORRECT",
                "reason": f"At least one chunk scored > {UPPER_TH}. Scores: {shown}"}
    if not good_docs:
        return {"good_docs": [], "verdict": "INCORRECT",
                "reason": f"No chunk scored > {LOWER_TH}. Scores: {shown}"}
    return {"good_docs": good_docs, "verdict": "AMBIGUOUS",
            "reason": f"Some chunks relevant, none > {UPPER_TH}. Scores: {shown}"}

## 8. Knowledge refinement (decompose, filter, recompose)
Even a relevant chunk contains sentences that do not help. Refinement splits the text into sentences, asks the LLM to keep or drop each one, and joins the kept sentences into a short, focused context. This follows the paper's decompose-then-recompose idea, but the filtering is done by an LLM instead of a trained model. The source of the text depends on the verdict:
- CORRECT: refine the good internal chunks
- INCORRECT: refine the web results only
- AMBIGUOUS: refine internal + web results together

In [8]:
def decompose_to_sentences(text: str) -> List[str]:
    text = re.sub(r"\s+", " ", text).strip()           # collapse whitespace/newlines
    sentences = re.split(r"(?<=[.!?])\s+", text)        # split after . ! ?
    return [s.strip() for s in sentences if len(s.strip()) > 20]  # drop tiny fragments

class KeepOrDrop(BaseModel):
    keep: bool

filter_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a strict relevance filter.\n"
     "Return keep=true only if the sentence directly helps answer the question.\n"
     "Use ONLY the sentence. Output JSON only."),
    ("human", "Question: {question}\n\nSentence:\n{sentence}"),
])
filter_chain = filter_prompt | fast_llm.with_structured_output(KeepOrDrop)

def refine(state: State) -> dict:
    verdict = state["verdict"]
    if verdict == "CORRECT":
        docs_to_use = state.get("good_docs", [])
    elif verdict == "INCORRECT":
        docs_to_use = state.get("web_docs", [])
    else:  # AMBIGUOUS -> combine internal + web evidence (as in the CRAG paper)
        docs_to_use = state.get("good_docs", []) + state.get("web_docs", [])

    context = "\n\n".join(d.page_content for d in docs_to_use)
    strips = decompose_to_sentences(context)
    if not strips:
        return {"strips": [], "kept_strips": [], "refined_context": ""}

    decisions = filter_chain.batch(
        [{"question": state["question"], "sentence": s} for s in strips],
        config={"max_concurrency": MAX_CONC},
    )
    kept = [s for s, d in zip(strips, decisions) if d.keep]
    return {"strips": strips, "kept_strips": kept, "refined_context": "\n".join(kept)}

## 9. Query rewrite and web search (Tavily)
Used when the verdict is INCORRECT or AMBIGUOUS. A full question is not ideal for a search engine, so the LLM first rewrites it into a short, keyword-style search query (6 to 14 words). Tavily then returns cleaned page content, which I wrap as documents.

One fix I had to add: the small model sometimes leaked internal control text into the rewritten query. The code now cuts anything after `<|` and limits the query to 14 words, so the search always receives a clean keyword phrase.

In [9]:
rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Rewrite the user's question into a concise web search query.\n"
     "- Use only keywords.\n"
     "- Keep it between 6 and 14 words.\n"
     "- If the question asks for recent information, include a time constraint like 'last 30 days'.\n"
     "- Return ONLY the search query.\n"
     "- Do not use quotes.\n"
     "- Do not explain anything."),
    ("human", "Question: {question}"),
])
rewrite_chain = rewrite_prompt | fast_llm | StrOutputParser()

def rewrite_query(state: State) -> dict:
    raw = rewrite_chain.invoke({"question": state["question"]})
    query = raw.split("<|")[0].strip().strip('"\'')   
    query = " ".join(query.split()[:14])              
    return {"web_query": query or state["question"]}

def web_search(state: State) -> dict:
    query = state.get("web_query") or state["question"]
    response = tavily.invoke({"query": query})
    results = response.get("results", []) if isinstance(response, dict) else []  # Tavily may return an error string
    web_docs = [
        Document(page_content=r["content"].strip(),
                 metadata={"title": r.get("title", ""), "url": r.get("url", "")})
        for r in results if r.get("content")
    ]
    return {"web_docs": web_docs}

## 10. Generate the final answer
The answer is generated only from the refined context. If nothing survived refinement, the node returns "I don't know" without calling the LLM, which saves a call and avoids guessing.

In [10]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer only using the provided context. If the answer is not present, say 'I don't know.'"),
    ("human", "Question: {question}\n\nContext:\n{refined_context}"),
])

def generate(state: State) -> dict:
    if not state.get("refined_context"):
        return {"answer": "I don't know. No relevant context survived refinement."}
    response = (prompt | llm).invoke({
        "question": state["question"],
        "refined_context": state["refined_context"],
    })
    return {"answer": response.content}

## 11. Router and graph
After the evaluator, CORRECT goes straight to refinement because the local documents are good enough. INCORRECT and AMBIGUOUS go through query rewriting and web search first. In the AMBIGUOUS case, refinement uses both the local chunks and the web results. The diagram printed below is generated from the code, so it always matches what the graph really does.

In [11]:
def route_after_eval(state: State) -> str:
    return "refine" if state["verdict"] == "CORRECT" else "rewrite_query"

graph = StateGraph(State)
graph.add_node("retrieve", retrieve)
graph.add_node("eval_each_doc", eval_each_doc_node)
graph.add_node("rewrite_query", rewrite_query)
graph.add_node("web_search", web_search)
graph.add_node("refine", refine)
graph.add_node("generate", generate)

graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "eval_each_doc")
graph.add_conditional_edges("eval_each_doc", route_after_eval,
                            {"refine": "refine", "rewrite_query": "rewrite_query"})
graph.add_edge("rewrite_query", "web_search")
graph.add_edge("web_search", "refine")
graph.add_edge("refine", "generate")
graph.add_edge("generate", END)

app = graph.compile()
print(app.get_graph().draw_mermaid())  # paste into README.md inside a ```mermaid block

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	retrieve(retrieve)
	eval_each_doc(eval_each_doc)
	rewrite_query(rewrite_query)
	web_search(web_search)
	refine(refine)
	generate(generate)
	__end__([<p>__end__</p>]):::last
	__start__ --> retrieve;
	eval_each_doc -.-> refine;
	eval_each_doc -.-> rewrite_query;
	refine --> generate;
	retrieve --> eval_each_doc;
	rewrite_query --> web_search;
	web_search --> refine;
	generate --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 12. Run
I ask the same questions as in the baseline notebook. For "What does agentic AI mean?", which is not in my PDF, every retrieved chunk scored 0.0, so the verdict was INCORRECT. The question was rewritten into the search query "what does agentic ai mean definition", Tavily returned web results, and the answer was generated from the refined web content. The baseline **[answered "I don't know" to the same question]**.

In [12]:
def ask(question: str) -> dict:
    result = app.invoke({"question": question})
    print("VERDICT :", result["verdict"])
    print("REASON  :", result["reason"])
    print("WEB QUERY:", result.get("web_query", "(not used)"))
    print("ANSWER  :", result["answer"])
    return result

In [13]:
question = "What does agentic ai means?"
result = ask(question)

VERDICT : INCORRECT
REASON  : No chunk scored > 0.3. Scores: [0.0, 0.0, 0.0, 0.0]
WEB QUERY: what does agentic ai mean definition
ANSWER  : Agentic AI refers to artificial‑intelligence systems that have agency – the ability to take initiative, reason, plan, and act independently toward a goal. Unlike traditional, purely reactive models that only give a classification or response when prompted, an agentic AI can proactively break down objectives, sequence multi‑step actions, use external tools (e.g., browsers, APIs, code), and adapt its strategy based on real‑time feedback, all with limited or no human oversight.
